Task 1
Sentiment Analysis
Ex.1. Basic Sentiment Analysis Using TextBlob Library
1. Choose 3 objects with reviews in English using google map (hotel, café, restaurant etc).
Extract 20 reviews as text. Present obtained date with final mark of review (number of stars) in
DateFrame. -
2. Perform Sentiment Analysis on obtained texts using TextBlob. Present obtained ‘polarity’ and
‘subjectivity’).
3. Compare results of Sentiment Analysis and Number of starts. Correct it if it is necessary.
4. Create three output files with two columns “Review” (original text) and “Sentiment” (0 or 1).


In [ ]:
!pip install python-docx

In [ ]:
import gdown
file_id = '1-t-xQtgTwsODTRB3zUjGtimUwNxoruWt'
url = f'https://docs.google.com/uc?id={file_id}'
output = 'Reviews.docx'
gdown.download(url, output, quiet=False)

Downloading...
From: https://docs.google.com/uc?id=1-t-xQtgTwsODTRB3zUjGtimUwNxoruWt
To: /content/Reviews.docx
100%|██████████| 1.62M/1.62M [00:00<00:00, 31.7MB/s]


'Reviews.docx'

With the use of table, checking, if everything is working

In [ ]:
from prettytable import PrettyTable
from docx import Document
import re

def extract_reviews_from_docx(file_path):
    doc = Document(file_path)
    reviews, loc, review = {}, None, None
    skip_first = True
    for p in doc.paragraphs:
        text = p.text.strip()
        if not text:
            continue
        if skip_first:
            skip_first = False
            continue
        #defining new location
        if not loc or len(reviews.get(loc, [])) >= 20:
            loc, reviews[loc], review = text, [], None
        else:
            # rating checking
            m = re.search(r'Number of stars:\s*(\d+)', text)
            if m:
                if review:
                    review['stars'] = int(m.group(1))
                    reviews[loc].append(review)
                    review = None
            else:
                # review text
                clean_text = re.sub(r'[★]+', '', text).strip()
                if clean_text:
                    review = {'text': clean_text, 'stars': 0}
    return reviews

file_path = 'Reviews.docx'
reviews_data = extract_reviews_from_docx(file_path)

table = PrettyTable()
table.field_names = ["Place", "Review", "Stars"]
for k in ["Place", "Review", "Stars"]:
    table.align[k] = "l" if k != "Stars" else "c"
table.max_width["Review"] = 60

for loc, reviews in reviews_data.items():
    for i, r in enumerate(reviews[:3]):
        txt = r['text'][:100] + ('...' if len(r['text']) > 100 else '')
        table.add_row([loc if i == 0 else "", txt, r['stars']])
    if loc != list(reviews_data.keys())[-1]:
        table.add_row(["-"*30, "-"*60, "-"*5])

print("General review:")
print(table)

General review:
+--------------------------------+--------------------------------------------------------------+-------+
| Place                          | Review                                                       | Stars |
+--------------------------------+--------------------------------------------------------------+-------+
| St. Basils Cathedral           | It is the icon of Russia - a very colorful building. We were |   4   |
|                                | supposed to visit the inside through th...                   |       |
|                                | St. Basil’s Cathedral is one of the most stunning and unique |   5   |
|                                | buildings I’ve ever seen. Located at th...                   |       |
|                                | One of the best tourist destinations/ attractions in Moscow  |   4   |
|                                | , our visit on may 2025 , the res Square...                  |       |
| ----------------------------

In [ ]:
def extract_reviews_with_ratings(file_path):
    doc = Document(file_path)
    reviews = []
    current_location = None
    current_review = None

    for para in doc.paragraphs:
        text = para.text.strip()
        if not text:
            continue

        # defining the location
        if text in ['St. Basils Cathedral', 'Big Ben', 'Eiffel Tower']:
            current_location = text
            continue

        # skipping the header
        if text.lower() == 'reviews':
            continue

        stars_match = re.search(r'(Number of stars:|★)\s*(\d+)', text)
        if stars_match and current_location:
            stars = int(stars_match.group(2))
            if current_review:
                current_review['stars'] = stars
                reviews.append(current_review)
                current_review = None
        elif current_location:
            clean_text = re.sub(r'★+', '', text).strip()
            if clean_text:
                current_review = {
                    'text': clean_text,
                    'location': current_location,
                    'stars': None
                }

    return reviews


In [ ]:
#extracting the reviews
file_path = 'Reviews.docx'
reviews_data = extract_reviews_from_docx(file_path)

In [ ]:
import pandas as pd
from textblob import TextBlob
# combining all the review for the analysis
all_reviews = []
for location, reviews in reviews_data.items():
    for review in reviews:
        all_reviews.append(review['text'])

In [ ]:
#creating DataFrame
df = pd.DataFrame({'text': all_reviews})

In [ ]:
# analysis
def analyze_sentiment(text):
    analysis = TextBlob(str(text))
    return analysis.sentiment.polarity, analysis.sentiment.subjectivity

df[['polarity', 'subjectivity']] = df['text'].apply(
    lambda x: pd.Series(analyze_sentiment(x))
)

In [ ]:
# creating a table with colimns
table = PrettyTable()
table.field_names = ["Review", "polarity", "Subjectivity"]
for index, row in df.head(5).iterrows():
    review_text = row['text']
    polarity = f"{row['polarity']:.2f}"
    subjectivity = f"{row['subjectivity']:.2f}"
    if len(review_text) > 50:
        review_text = review_text[:50] + '...'
    table.add_row([review_text, polarity, subjectivity])

print("\nfirst 5 reviews:")
print(table)


first 5 reviews:
+-------------------------------------------------------+----------+--------------+
|                         Review                        | polarity | Subjectivity |
+-------------------------------------------------------+----------+--------------+
| It is the icon of Russia - a very colorful buildin... |   0.02   |     0.18     |
| St. Basil’s Cathedral is one of the most stunning ... |   0.07   |     0.42     |
| One of the best tourist destinations/ attractions ... |   0.11   |     0.31     |
| I loved the center of Moscow, the Red Square and S... |   0.18   |     0.34     |
| St. Basil's Cathedral, located in Moscow's Red Squ... |   0.14   |     0.47     |
+-------------------------------------------------------+----------+--------------+


In [ ]:
reviews = extract_reviews_with_ratings('Reviews.docx')

In [ ]:
# preparing data for analysis
data = []
for location, reviews in reviews_data.items():
    for review in reviews:
        if 'stars' in review:
            data.append({
                'location': location,
                'text': review['text'],
                'stars': review['stars'],
                'sentiment': analyze_sentiment(review['text'])
            })

df = pd.DataFrame(data)

In [ ]:
df = pd.DataFrame(reviews)

In [ ]:
def analyze_sentiment(text):
    return 1 if TextBlob(text).sentiment.polarity > 0 else 0

reviews = extract_reviews_with_ratings('Reviews.docx')

if not reviews:
    print("Error. Nothing was found.")
else:
    df = pd.DataFrame(reviews)
    df = df.dropna(subset=['stars'])
    df['sentiment'] = df['text'].apply(analyze_sentiment)

    df['rating_match'] = ((df['stars'] >= 4) & (df['sentiment'] == 1)) | ((df['stars'] < 4) & (df['sentiment'] == 0))

    for location in ['St. Basils Cathedral', 'Big Ben', 'Eiffel Tower']:
        location_reviews = df[df['location'] == location]
        filename = f"{location.replace(' ', '_')}_sentiment.csv"

        if not location_reviews.empty:
            location_reviews[['text', 'sentiment']].rename(
                columns={'text': 'Review', 'sentiment': 'Sentiment'}
            ).to_csv(filename, index=False)
            print(f"{filename} (reviews: {len(location_reviews)})")

            # example for a location
            print(f"Review example({location}, {location_reviews.iloc[0]['stars']} stars):")
            print(location_reviews.iloc[0]['text'][:100] + "...")
            print(f"Tonality: {location_reviews.iloc[0]['sentiment']}")
            print(f"Match: {'Yes' if location_reviews.iloc[0]['rating_match'] else 'No'}\n")
        else:
            print(f"No reviews for location: {location}")

    # general statistics
    print("\ngeneral statistics:")
    print(f"general number of reviews: {len(df)}")
    print(f"location disrtibution:\n{df['location'].value_counts()}")
    print(f"\match of a review and rating: {df['rating_match'].mean():.1%}")

    table = PrettyTable()
    table.field_names = ["Review", "Stars", "Tonality", "Match"]

    for _, row in df.head().iterrows():
        text = row['text'][:50] + '...' if len(row['text']) > 50 else row['text']
        table.add_row([text, row['stars'], row['sentiment'], 'Yes' if row['rating_match'] else 'No'])

    print("\Table of the first five reviews:")
    print(table)


    mismatches = df[~df['rating_match']]
    if not mismatches.empty:
        print("\nMatching examples:")
        mismatch_table = PrettyTable()
        mismatch_table.field_names = ["Review", "Stars", "Tonality", "Cause"]

        for _, row in mismatches.head(3).iterrows():
            text = row['text'][:50] + '...' if len(row['text']) > 50 else row['text']
            reason = "High rating, but negative review" if row['stars'] >=4 and row['sentiment'] == 0 else "Low rating, but positive review"
            mismatch_table.add_row([text, row['stars'], row['sentiment'], reason])

        print(mismatch_table)
    else:
        print("\nAll the reviews match ratings!")

St._Basils_Cathedral_sentiment.csv (reviews: 20)
Review example(St. Basils Cathedral, 4 stars):
It is the icon of Russia - a very colorful building. We were supposed to visit the inside through th...
Tonality: 1
Match: Yes

Big_Ben_sentiment.csv (reviews: 20)
Review example(Big Ben, 4 stars):
Big Ben is one of London’s most iconic landmarks and a must see for any visitor! completed in 1859 w...
Tonality: 1
Match: Yes

Eiffel_Tower_sentiment.csv (reviews: 19)
Review example(Eiffel Tower, 5 stars):
Eiffel Tower: An Imposing Symbol with History and a Magical Atmosphere The Eiffel Tower (Tour Eiffel...
Tonality: 1
Match: Yes


general statistics:
general number of reviews: 59
location disrtibution:
location
St. Basils Cathedral    20
Big Ben                 20
Eiffel Tower            19
Name: count, dtype: int64
\match of a review and rating: 93.2%
\Table of the first five reviews:
+-------------------------------------------------------+-------+----------+-------+
|                       

Ex.2.1 Training a Sentiment Model Using TFIDF and Logistic Regression
1. Build a simple sentiment analysis model with TFIDF vectorization and logistic regression
learning with datasets from gethub examples. (amazon_labelled.txt, yelp_labelled.txt,
imdb_labelled.txt)

In [ ]:
import pandas as pd
import gdown
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [ ]:
import pandas as pd
import gdown
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

file_ids_unique = [
    '1Dt1fKOvXVNUi01A0IeL7-DHQjWekBAzA',
    '1V4SNhTua9nj46fLWdmZ_P_JadYMq36S7',
    '1sBhTEVJ0yPfQHSnab5VyxPVYmsvCGfTN'
]

local_files_unique = [
    'yelp_labelled.txt',
    'imdb_labelled.txt',
    'amazon_labelled.txt'
]


for file_id, filename in zip(file_ids_unique, local_files_unique):
    url = f'https://drive.google.com/uc?id={file_id}'
    gdown.download(url, filename, quiet=False)


dataframes_unique = []
for filename in local_files_unique:
    df = pd.read_csv(filename, sep='\t', header=None, names=['review', 'label'])
    dataframes_unique.append(df)


data_unique = pd.concat(dataframes_unique, ignore_index=True)
print(f"Number of samples: {len(data_unique)}")
print(data_unique.head(3))

Downloading...
From: https://drive.google.com/uc?id=1Dt1fKOvXVNUi01A0IeL7-DHQjWekBAzA
To: /content/yelp_labelled.txt
100%|██████████| 61.3k/61.3k [00:00<00:00, 32.1MB/s]
Downloading...
From: https://drive.google.com/uc?id=1V4SNhTua9nj46fLWdmZ_P_JadYMq36S7
To: /content/imdb_labelled.txt
100%|██████████| 85.3k/85.3k [00:00<00:00, 20.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1sBhTEVJ0yPfQHSnab5VyxPVYmsvCGfTN
To: /content/amazon_labelled.txt
100%|██████████| 58.2k/58.2k [00:00<00:00, 62.2MB/s]

Number of samples: 2748
                                      review  label
0                   Wow... Loved this place.      1
1                         Crust is not good.      0
2  Not tasty and the texture was just nasty.      0


In [ ]:
X_unique = data_unique['review']
y_unique = data_unique['label'].astype(int)

In [ ]:
X_train_unique, X_test_unique, y_train_unique, y_test_unique = train_test_split(
    X_unique, y_unique, test_size=0.2, random_state=42
)


In [ ]:
vectorizer_unique = TfidfVectorizer()
X_train_tfidf_unique = vectorizer_unique.fit_transform(X_train_unique)
X_test_tfidf_unique = vectorizer_unique.transform(X_test_unique)

In [ ]:
model_unique = LogisticRegression(max_iter=1000)
model_unique.fit(X_train_tfidf_unique, y_train_unique)

LogisticRegression(max_iter=1000)

In [ ]:
y_pred_unique = model_unique.predict(X_test_tfidf_unique)
accuracy_unique = accuracy_score(y_test_unique, y_pred_unique)
print(f"Model accuracy: {accuracy_unique:.2%}")

Model accuracy: 82.00%


The model shows good accuracy on standard datasets, but most likely, its effectiveness may decrease on highly specialized or less formalized texts.

In [ ]:
X_test_unique_reset = X_test_unique.reset_index(drop=True)
y_test_unique_reset = y_test_unique.reset_index(drop=True)
y_pred_unique_reset = y_pred_unique
for i in range(5):
    if y_test_unique_reset[i] != y_pred_unique_reset[i]:
        print(f"\nReview: {X_test_unique_reset[i]}")
        print(f"Real class: {y_test_unique_reset[i]}")
        print(f"Predicted class: {y_pred_unique_reset[i]}")


Review: This product is very High quality Chinese CRAP!!!!!!
Real class: 0
Predicted class: 1


Check model accuracy with examples from from ex.1. Additionally present 2-3 examples,
where model will give incorrect result.

In [ ]:
file_path = 'Reviews.docx'
reviews_df = extract_reviews_from_docx(file_path)

In [ ]:
import pandas as pd
data = []
for location, reviews in reviews_df.items():
    for review in reviews:
        data.append({
            'location': location,
            'text': review['text'],
            'stars': review['stars']
        })

df_ex1 = pd.DataFrame(data)

df_ex1['sentiment'] = df_ex1['stars'].apply(lambda x: 1 if x >= 4 else 0)

print(df_ex1.head())

               location                                               text  \
0  St. Basils Cathedral  It is the icon of Russia - a very colorful bui...   
1  St. Basils Cathedral  St. Basil’s Cathedral is one of the most stunn...   
2  St. Basils Cathedral  One of the best tourist destinations/ attracti...   
3  St. Basils Cathedral  I loved the center of Moscow, the Red Square a...   
4  St. Basils Cathedral  St. Basil's Cathedral, located in Moscow's Red...   

   stars  sentiment  
0      4          1  
1      5          1  
2      4          1  
3      3          0  
4      5          1  


In [ ]:
X_ex1 = vectorizer_unique.transform(df_ex1['text'])

In [ ]:
y_pred_ex1 = model_unique.predict(X_ex1)

In [ ]:
accuracy_ex1 = accuracy_score(df_ex1['sentiment'], y_pred_ex1)
print(f"Accuracy on Ex.1 reviews: {accuracy_ex1:.2%}")

Accuracy on Ex.1 reviews: 76.27%


In [ ]:
misclassified = df_ex1[df_ex1['sentiment'] != y_pred_ex1]
print("\nModel Mistakes:")
for i, row in misclassified.head(3).iterrows():
    print(f"\\nReview: {row['text'][:100]}...")
    print(f"True: {row['sentiment']} (Stars: {row['stars']}), Predicted: {y_pred_ex1[i]}")


Model Mistakes:
\nReview: It is the icon of Russia - a very colorful building. We were supposed to visit the inside through th...
True: 1 (Stars: 4), Predicted: 0
\nReview: I loved the center of Moscow, the Red Square and Saint Basil's Cathedral. I used to teach in Tverska...
True: 0 (Stars: 3), Predicted: 1
\nReview: The entrance fee of 2000 rubles for foreigners is totally excessive compared to the price for Russia...
True: 1 (Stars: 4), Predicted: 0


Build a simple sentiment analysis model with TFIDF vectorization and logistic regression learning with three datasets obtained in Ex. 1.

In [ ]:
X = df_ex1['text']
y = df_ex1['sentiment']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
vectorizer_ex1 = TfidfVectorizer(max_features=5000)
X_train_vect = vectorizer_ex1.fit_transform(X_train)
X_test_vect = vectorizer_ex1.transform(X_test)

In [ ]:
model_ex1 = LogisticRegression(max_iter=1000)
model_ex1.fit(X_train_vect, y_train)


LogisticRegression(max_iter=1000)

In [ ]:
y_pred = model_ex1.predict(X_test_vect)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model accuracy (Ex.2.2): {accuracy:.2%}")

Model accuracy (Ex.2.2): 83.33%


In [ ]:
misclassified = X_test[y_test != y_pred]
print("\nErrors examples:")
for i in range(min(3, len(misclassified))):
    print(f"\\nReview: {misclassified.iloc[i][:100]}...")
    print(f"True: {y_test.iloc[i]}, Predicted: {y_pred[i]}")


Errors examples:
\nReview: Also difficult to put on.I'd recommend avoiding this product....
True: 0, Predicted: 1
\nReview: Comfortable fit - you need your headset to be comfortable for at least an hour at a time, if not for...
True: 1, Predicted: 1
\nReview: Tried to go here for lunch and it was a madhouse....
True: 0, Predicted: 0


2. Check model accuracy with examples from lection, and proposed 3 new examples.
Additionally present 2-3 examples, where model will give incorrect result.

In [ ]:
def extract_reviews_from_docx(file_path):
    doc = Document(file_path)
    reviews_data = []
    current_location = None
    current_review = None

    for para in doc.paragraphs:
        text = para.text.strip()
        if not text:
            continue
        if text in ['St. Basils Cathedral', 'Big Ben', 'Eiffel Tower']:
            current_location = text
            continue

        if text.lower() == 'reviews':
            continue

        stars_match = re.search(r'(Number of stars:|★)\s*(\d+)', text)
        if stars_match and current_location:
            stars = int(stars_match.group(2))
            if current_review:
                current_review['stars'] = stars
                current_review['location'] = current_location
                reviews_data.append(current_review)
                current_review = None
        elif current_location:
            clean_text = re.sub(r'★+', '', text).strip()
            if clean_text:
                current_review = {'text': clean_text, 'stars': None}

    return reviews_data

file_path = 'Reviews.docx'
reviews_data = extract_reviews_from_docx(file_path)
df_reviews = pd.DataFrame(reviews_data)
df_reviews = df_reviews.dropna(subset=['stars'])
df_reviews['sentiment'] = df_reviews['stars'].apply(lambda x: 1 if x >=4 else 0)

file_ids = [
    '1Dt1fKOvXVNUi01A0IeL7-DHQjWekBAzA',
    '1V4SNhTua9nj46fLWdmZ_P_JadYMq36S7',
    '1sBhTEVJ0yPfQHSnab5VyxPVYmsvCGfTN'
]
filenames = ['yelp_labelled.txt', 'imdb_labelled.txt', 'amazon_labelled.txt']


In [ ]:
datasets = []
for file_id, filename in zip(file_ids, filenames):
    url = f'https://drive.google.com/uc?id={file_id}'
    gdown.download(url, filename, quiet=False)
    df = pd.read_csv(filename, sep='\t', header=None, names=['text', 'sentiment'])
    datasets.append(df)

Downloading...
From: https://drive.google.com/uc?id=1Dt1fKOvXVNUi01A0IeL7-DHQjWekBAzA
To: /content/yelp_labelled.txt
100%|██████████| 61.3k/61.3k [00:00<00:00, 57.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=1V4SNhTua9nj46fLWdmZ_P_JadYMq36S7
To: /content/imdb_labelled.txt
100%|██████████| 85.3k/85.3k [00:00<00:00, 38.4MB/s]
Downloading...
From: https://drive.google.com/uc?id=1sBhTEVJ0yPfQHSnab5VyxPVYmsvCGfTN
To: /content/amazon_labelled.txt
100%|██████████| 58.2k/58.2k [00:00<00:00, 15.6MB/s]


In [ ]:
all_data = pd.concat([df_reviews[['text', 'sentiment']]] + datasets, ignore_index=True)

In [ ]:
X = all_data['text']
y = all_data['sentiment']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)
y_pred = model.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)
print(f"general model accuracy: {accuracy:.2%}")

general model accuracy: 79.72%


In [ ]:
X_reviews = vectorizer.transform([r['text'] for r in reviews_data])
y_reviews_true = [1 if r['stars'] >= 4 else 0 for r in reviews_data]
y_reviews_pred = model.predict(X_reviews)
accuracy_reviews = accuracy_score(y_reviews_true, y_reviews_pred)
print(f"Accuracy on reviews: {accuracy_reviews:.2%}")

Accuracy on reviews: 93.22%


A comparison with the rating (stars) showed that in 93.2% of cases, the tone of the Text Blob coincided with the user's rating. This indicates a high consistency between automatic analysis and manual estimates.

Text Blob does a good job of analyzing tonality, but errors are possible in cases where the context is not obvious

Overall accuracy of the model: 79.72%.

Accuracy based on reviews from Ex.1: 93.22%, which is significantly higher than that of a model trained only on standard datasets. This suggests that the inclusion of specific data improves the quality of predictions for similar texts.

Conclusion: Combining data from different sources can improve the accuracy of the model, especially if the target texts have unique features.